# Чекпоинт 7 - наблюдаемость модели (MLflow + S3)

Модель: **CatBoost на «умных негативах»** (site-selection для размещения банкоматов)
Финальная версия зафиксирована в Model Registry с тегом/алиасом **PRD**

**Инфраструктура (поднята отдельно, см. `docker-compose.yml`):**
- локальный **MLflow** в Docker-контейнере;
- локальный **S3** = MinIO (бакет `mlflow`);
- блокнот подключён к MLflow по `http://localhost:5000`

## Подготовка среды и воспроизводимость (пункт 5)
Подключение блокнота к MLflow, фиксация seed и параметров финальной модели

In [1]:
import os, random
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import LeaveOneGroupOut
from scipy.spatial import cKDTree
import mlflow

# воспроизводимость (пункт 5 чекпоинта)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# подключение к нашему MLflow-серверу в Docker
mlflow.set_tracking_uri("http://localhost:5000")
EXPERIMENT_NAME = "atm_placement"
mlflow.set_experiment(EXPERIMENT_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())

# конфиг финальной модели
DATA_PATH = "grid_features.csv"
REGISTERED_MODEL_NAME = "atm_placement_catboost"

FINAL_PARAMS = dict(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    auto_class_weights="Balanced",
    eval_metric="AUC",
    random_seed=SEED,
    verbose=False)

EARLY_STOPPING_ROUNDS = 50
print("Конфиг готов")

2026/06/07 13:23:38 INFO mlflow.tracking.fluent: Experiment with name 'atm_placement' does not exist. Creating a new experiment.


Tracking URI: http://localhost:5000
Конфиг готов


## Данные и стратегия негативов (пункт 5)
Загрузка `grid_features.csv`, формирование таргета и **умных негативов** (баланс 1:1: 10% случайных / 30% возле инфраструктуры / 60% соседних по координатам). Это описывает данные и способ их получения для воспроизводимости

In [2]:
# Данные + умные негативы (10% random / 30% негативы возле инфры / 60% соседи)
df = pd.read_csv(DATA_PATH)
df["target"] = (df["atm_count"] > 0).astype(int)

positives = df[df["target"] == 1]
negatives = df[df["target"] == 0]

n_pos = len(positives)
n_neg = n_pos # баланс 1:1
n_random = int(n_neg * 0.10)
n_infra = int(n_neg * 0.30)
n_neighbors = n_neg - n_random - n_infra
print(f"Позитивы: {n_pos}")
print(f"Негативы: {n_neg} (random={n_random}, infra={n_infra}, neighbors={n_neighbors})")

np.random.seed(SEED)

# 1) случайные негативы
random_neg = negatives.sample(n=n_random, random_state=SEED)

# 2) негативы возле инфраструктуры (POI есть, банкомата нет)
infra_mask = (negatives["total_poi_500m"] > 3) & (negatives["orgs_500m"] > 5)
infra_candidates = negatives[infra_mask]
print(f"Кандидатов infra: {len(infra_candidates)}")
infra_neg = infra_candidates.sample(n=min(n_infra, len(infra_candidates)), random_state=SEED)

# 3) соседние к позитивам ячейки без банкомата (по координатам)
pos_coords = positives[["lat", "lon"]].values
neg_coords = negatives[["lat", "lon"]].values
tree = cKDTree(neg_coords)
k_per_pos = max(1, n_neighbors // n_pos + 1)
_, indices = tree.query(pos_coords, k=k_per_pos)
neighbor_indices = np.unique(indices.ravel())
np.random.shuffle(neighbor_indices)
neighbor_indices = neighbor_indices[:n_neighbors]
neighbor_neg = negatives.iloc[neighbor_indices]

# объединяем негативы без дублей + позитивы, перемешиваем
all_neg_idx = set(random_neg.index) | set(infra_neg.index) | set(neighbor_neg.index)
sampled_negatives = negatives.loc[list(all_neg_idx)]
df_sampled = pd.concat([positives, sampled_negatives]).sample(frac=1, random_state=SEED)

# фичи: выкидываем утечку + координаты + служебные
drop_cols = ['atm_count', 'sber_count', 'tinkoff_count', 'vtb_count',
             'alfa_count', 'gazprom_count', 'raiff_count',
             'lat', 'lon', 'target']
feature_cols = [c for c in df_sampled.columns if c not in drop_cols and c != 'city']

X = df_sampled[feature_cols].fillna(0).copy()
y = df_sampled["target"].values
groups = df_sampled["city"].values

print(f"\nИтоговая выборка: {len(df_sampled)} строк, target rate: {df_sampled['target'].mean():.4f}")
print(f"Число фичей: {len(feature_cols)}")
print(df_sampled.groupby('city')['target'].agg(['mean', 'sum', 'count']))

Позитивы: 6264
Негативы: 6264 (random=626, infra=1879, neighbors=3759)
Кандидатов infra: 7982

Итоговая выборка: 11892 строк, target rate: 0.5267
Число фичей: 90
                     mean   sum  count
city                                  
Казань           0.507246   350    690
Москва           0.553313  3674   6640
Нижний Новгород  0.449126   437    973
Новосибирск      0.444444   520   1170
Санкт-Петербург  0.530384  1283   2419


## Переобучение модели (пункт 2 чекпоинита) и метрики train/validation/test (пункт 3)
LOGO-кросс-валидация по городам (воспроизводит метрики из model card) + явный train/val/test сплит на отложенном городе для чистой тройки метрик

In [3]:
from sklearn.model_selection import train_test_split

def roc_pr(y_true, proba):
    roc = roc_auc_score(y_true, proba)
    pr = average_precision_score(y_true, proba)
    base = y_true.mean()
    return roc, pr, (pr / base if base > 0 else 0.0)

# 1) LOGO-CV по городам
logo = LeaveOneGroupOut()
per_city = []
cv_train_roc, cv_train_pr = [], []
cv_test_roc, cv_test_pr, cv_test_lift = [], [], []

for tr_idx, te_idx in logo.split(X, y, groups):
    city = groups[te_idx[0]]
    m = CatBoostClassifier(**FINAL_PARAMS)
    m.fit(X.iloc[tr_idx], y[tr_idx],
          eval_set=(X.iloc[te_idx], y[te_idx]),
          early_stopping_rounds=EARLY_STOPPING_ROUNDS)
    p_tr = m.predict_proba(X.iloc[tr_idx])[:, 1]
    p_te = m.predict_proba(X.iloc[te_idx])[:, 1]
    tr_roc, tr_pr, _ = roc_pr(y[tr_idx], p_tr)
    te_roc, te_pr, te_lift = roc_pr(y[te_idx], p_te)
    cv_train_roc.append(tr_roc); cv_train_pr.append(tr_pr)
    cv_test_roc.append(te_roc); cv_test_pr.append(te_pr); cv_test_lift.append(te_lift)
    per_city.append({'city': city, 'roc_auc': te_roc, 'pr_auc': te_pr, 'pr_over_baseline': te_lift})
    print(f"{city}: ROC-AUC={te_roc:.4f}, PR-AUC={te_pr:.4f}, PR/baseline={te_lift:.2f}x")

per_city_df = pd.DataFrame(per_city)
print("\nCV СРЕДНИЕ:  ROC-AUC=%.4f  PR-AUC=%.4f  PR/baseline=%.2fx" %
      (np.mean(cv_test_roc), np.mean(cv_test_pr), np.mean(cv_test_lift)))

# 2) Явный train/val/test сплит
TEST_CITY = "Санкт-Петербург"
is_test = (groups == TEST_CITY)
X_tr_all, y_tr_all = X[~is_test], y[~is_test]
X_test_city, y_test_city = X[is_test], y[is_test]
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr_all, y_tr_all, test_size=0.2, random_state=SEED, stratify=y_tr_all)

model_split = CatBoostClassifier(**FINAL_PARAMS)
model_split.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=EARLY_STOPPING_ROUNDS)

s_train = roc_pr(y_tr, model_split.predict_proba(X_tr)[:, 1])
s_val   = roc_pr(y_val, model_split.predict_proba(X_val)[:, 1])
s_test  = roc_pr(y_test_city, model_split.predict_proba(X_test_city)[:, 1])
print(f"\nSPLIT (отложенный город = {TEST_CITY}):")
print(f"  train: ROC={s_train[0]:.4f}  PR={s_train[1]:.4f}")
print(f"  val: ROC={s_val[0]:.4f}  PR={s_val[1]:.4f}")
print(f"  test: ROC={s_test[0]:.4f}  PR={s_test[1]:.4f}  lift={s_test[2]:.2f}x")

Казань: ROC-AUC=0.8700, PR-AUC=0.8665, PR/baseline=1.71x
Москва: ROC-AUC=0.8615, PR-AUC=0.8840, PR/baseline=1.60x
Нижний Новгород: ROC-AUC=0.8742, PR-AUC=0.8525, PR/baseline=1.90x
Новосибирск: ROC-AUC=0.9050, PR-AUC=0.8859, PR/baseline=1.99x
Санкт-Петербург: ROC-AUC=0.8558, PR-AUC=0.8600, PR/baseline=1.62x

CV СРЕДНИЕ:  ROC-AUC=0.8733  PR-AUC=0.8698  PR/baseline=1.76x

SPLIT (отложенный город = Санкт-Петербург):
  train: ROC=0.9310  PR=0.9429
  val: ROC=0.8716  PR=0.8855
  test: ROC=0.8564  PR=0.8606  lift=1.62x


### Логирование параметров и метрик в MLflow (пункт 3)
Открываем ран `final_smart_negatives`, пишем гиперпараметры, параметры воспроизводимости и метрики (CV-средние + train/val/test сплита)

In [4]:
with mlflow.start_run(run_name="final_smart_negatives") as run:
    RUN_ID = run.info.run_id

    # параметры модели + воспроизводимость
    mlflow.log_params(FINAL_PARAMS)
    mlflow.log_params({
        "seed": SEED,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        "n_samples": len(df_sampled),
        "n_features": len(feature_cols),
        "neg_strategy": "smart_1to1_10random_30infra_60neighbors",
        "test_city_for_split": TEST_CITY,
        "data_path": DATA_PATH})

    mlflow.set_tags({"model_type": "catboost", "strategy": "smart_negatives", "status": "candidate"})

    # метрики LOGO-CV (headline, как в README)
    mlflow.log_metrics({
        "cv_test_roc_auc": float(np.mean(cv_test_roc)),
        "cv_test_pr_auc": float(np.mean(cv_test_pr)),
        "cv_test_pr_over_baseline": float(np.mean(cv_test_lift)),
        "cv_train_roc_auc": float(np.mean(cv_train_roc)),
        "cv_train_pr_auc": float(np.mean(cv_train_pr))})

    # метрики train/val/test сплита
    mlflow.log_metrics({
        "split_train_roc_auc": s_train[0], "split_train_pr_auc": s_train[1],
        "split_val_roc_auc": s_val[0],     "split_val_pr_auc": s_val[1],
        "split_test_roc_auc": s_test[0],   "split_test_pr_auc": s_test[1],
        "split_test_pr_over_baseline": s_test[2]})

print("RUN_ID:", RUN_ID)
print("Готово")

🏃 View run final_smart_negatives at: http://localhost:5000/#/experiments/2/runs/1826b14df2e94625aaf95f55c4260603
🧪 View experiment at: http://localhost:5000/#/experiments/2
RUN_ID: 1826b14df2e94625aaf95f55c4260603
Готово


## Финальная (PRD) модель + графики и примеры предсказаний (пункты 2, 4)
Обучаем финальную модель на всей собранной выборке и готовим артефакты: confusion matrix, learning curve, feature importance, примеры предсказаний

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

os.makedirs("artifacts", exist_ok=True)

# 1) ФИНАЛЬНАЯ (PRD) модель: учим на ВСЕЙ собранной выборке
final_model = CatBoostClassifier(**FINAL_PARAMS)
final_model.fit(X, y) # все города, все строки
print("Финальная модель обучена на", len(X), "строках")

# 2) Confusion matrix (на отложенном городе, по model_split)
proba_test = model_split.predict_proba(X_test_city)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)
cm = confusion_matrix(y_test_city, pred_test)
fig_cm, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["нет ATM", "есть ATM"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion matrix (test = {TEST_CITY}, thr=0.5)")
fig_cm.tight_layout(); fig_cm.savefig("artifacts/confusion_matrix.png", dpi=120); plt.close(fig_cm)

# 3) Learning curve (train vs validation AUC по итерациям)
ev = model_split.get_evals_result()
fig_lc, ax = plt.subplots(figsize=(6, 4))
for split_name, metrics in ev.items():
    if "AUC" in metrics:
        ax.plot(metrics["AUC"], label=split_name)
ax.set_xlabel("итерация"); ax.set_ylabel("AUC"); ax.set_title("Learning curve"); ax.legend()
fig_lc.tight_layout(); fig_lc.savefig("artifacts/learning_curve.png", dpi=120); plt.close(fig_lc)

# 4) Feature importance (топ-20)
fi = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
fi.to_csv("artifacts/feature_importance.csv", header=["importance"])
fig_fi, ax = plt.subplots(figsize=(7, 6))
fi.head(20).iloc[::-1].plot(kind="barh", ax=ax)
ax.set_title("Feature importance (top-20)")
fig_fi.tight_layout(); fig_fi.savefig("artifacts/feature_importance.png", dpi=120); plt.close(fig_fi)

# 5) Примеры предсказаний (отложенный город)
examples = df_sampled.loc[X_test_city.index, ["city", "lat", "lon", "target"]].copy()
examples["proba"] = proba_test
examples["pred"] = pred_test
examples = examples.sort_values("proba", ascending=False)
examples.to_csv("artifacts/example_predictions.csv", index=False)

print("Артефакты сохранены в папку artifacts/:")
print(os.listdir("artifacts"))

Финальная модель обучена на 11892 строках
Артефакты сохранены в папку artifacts/:
['confusion_matrix.png', 'example_predictions.csv', 'feature_importance.csv', 'feature_importance.png', 'learning_curve.png']


### Сохранение модели и артефактов в S3 (пункт 4)
Заливаем в тот же ран (через MLflow в MinIO/S3) обученную модель, графики и примеры предсказаний, модель регистрируется в Model Registry

In [6]:
import mlflow.catboost
from mlflow.models import infer_signature

with mlflow.start_run(run_id=RUN_ID): # дописываем в существующий ран
    # графики и таблицы в S3
    mlflow.log_artifacts("artifacts", artifact_path="plots_and_examples")

    # сама финальная модель в S3 (+ регистрация в Model Registry)
    signature = infer_signature(X, final_model.predict_proba(X)[:, 1])
    mlflow.catboost.log_model(
        final_model,
        artifact_path="model",
        signature=signature,
        input_example=X.head(3),
        registered_model_name=REGISTERED_MODEL_NAME)

print("Модель и артефакты залиты в MLflow/S3")
print("Зарегистрирована модель:", REGISTERED_MODEL_NAME)

c:\Users\Dmitry\checkpoint7\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'atm_placement_catboost'.
2026/06/07 13:51:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: atm_placement_catboost, version 1
C

🏃 View run final_smart_negatives at: http://localhost:5000/#/experiments/2/runs/1826b14df2e94625aaf95f55c4260603
🧪 View experiment at: http://localhost:5000/#/experiments/2
Модель и артефакты залиты в MLflow/S3
Зарегистрирована модель: atm_placement_catboost


## Анализ ошибок модели (пункт 6)
Разбор ошибок на отложенном городе: категории ошибок (FP/FN и сильная/слабая инфра) и 16 самых уверенных примеров для ручного разбора

In [7]:
# пороги сильной инфры по обучающей выборке
poi_hi = X["total_poi_500m"].quantile(0.75)
org_hi = X["orgs_500m"].quantile(0.75)

err = df_sampled.loc[X_test_city.index, ["city", "lat", "lon", "target",
                                         "total_poi_500m", "orgs_500m"]].copy()
err["proba"] = model_split.predict_proba(X_test_city)[:, 1]
err["pred"] = (err["proba"] >= 0.5).astype(int)
err = err[err["pred"] != err["target"]].copy() # только ошибки

def categorize(r):
    strong = (r["total_poi_500m"] >= poi_hi) or (r["orgs_500m"] >= org_hi)
    if r["target"] == 0: # FP: модель сказала есть банкомат, по факту нет
        return "FP: сильная инфра без банкомата" if strong else "FP: ложное срабатывание в слабой зоне"
    else: # FN: модель сказала нет банкомата, по факту есть
        return "FN: банкомат в слабой зоне (спец-локация)" if not strong else "FN: пропуск в сильной зоне (близко к порогу)"

err["error_type"] = err.apply(categorize, axis=1)
err["confidence"] = (err["proba"] - 0.5).abs() # насколько модель была уверена в ошибке

print("Всего ошибок на тесте:", len(err), "из", len(X_test_city),
      f"(ошибок {len(err)/len(X_test_city):.1%})")
print("\nРаспределение по категориям:")
print(err["error_type"].value_counts())

# 16 самых уверенных ошибок для ручного разбора
top_err = err.sort_values("confidence", ascending=False).head(16)
top_err.to_csv("artifacts/error_analysis.csv", index=False)
print("\nТоп самых уверенных ошибок:")
print(top_err[["city", "target", "pred", "proba", "total_poi_500m", "orgs_500m", "error_type"]].to_string())

Всего ошибок на тесте: 542 из 2419 (ошибок 22.4%)

Распределение по категориям:
error_type
FN: банкомат в слабой зоне (спец-локация)       263
FP: сильная инфра без банкомата                 129
FP: ложное срабатывание в слабой зоне            97
FN: пропуск в сильной зоне (близко к порогу)     53
Name: count, dtype: int64

Топ самых уверенных ошибок:
                   city  target  pred     proba  total_poi_500m  orgs_500m                                 error_type
142588  Санкт-Петербург       0     1  0.976841           273.0      267.0            FP: сильная инфра без банкомата
142895  Санкт-Петербург       0     1  0.976706           208.0      139.0            FP: сильная инфра без банкомата
147160  Санкт-Петербург       0     1  0.973077            64.0      115.0            FP: сильная инфра без банкомата
149928  Санкт-Петербург       0     1  0.969430           119.0      144.0            FP: сильная инфра без банкомата
144111  Санкт-Петербург       0     1  0.966455         

### Сохранение разбора ошибок (пункт 6)
Текстовая интерпретация (типичные категории, причины, оценка корректируемости) сохраняется как артефакт в MLflow

In [8]:
error_summary = f"""# Анализ ошибок модели (пункт 6)

Тест: отложенный город {TEST_CITY}. Порог классификации 0.5.
Всего ошибок: {len(err)} из {len(X_test_city)} ({len(err)/len(X_test_city):.1%}).

## Категории ошибок
{err['error_type'].value_counts().to_string()}

## Интерпретация
1. FP сильная инфра без банкомата - все самые уверенные ошибки (proba 0.93 - 0.98).
   Место по признакам идеально под банкомат, но его нет. Причины вне данных:
   аренда, насыщенность, договоры, бизнес-решения банка. Некорректируемо при текущих фичах.
2. FN банкомат в слабой зоне (спец-локация) - самая массовая группа.
   Банкомат в месте с бедной инфраструктурой (вокзал, завод, ТЦ на отшибе).
   Грид такие объекты не описывает, поэтому в основном некорректируемо.
3. FP ложное срабатывание в слабой зоне - частично корректируемо порогом/фичами.
4. FN пропуск в сильной зоне - у границы 0.5, корректируемо настройкой порога.

## Вывод
Приблизительно 72% ошибок (категории 1-2) некорректируемы: целевая разметка зависит от факторов
вне грида. Приблизительно 28% (категории 3-4) корректируемы порогом и обогащением признаков
(трафик, население, конкуренты, тип объекта, и т.п.). Дообучение под них даст выигрыш в пределах
шума при риске переобучения под один город"""

with open("artifacts/error_analysis_summary.md", "w", encoding="utf-8") as f:
    f.write(error_summary)

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact("artifacts/error_analysis.csv", artifact_path="error_analysis")
    mlflow.log_artifact("artifacts/error_analysis_summary.md", artifact_path="error_analysis")
    mlflow.log_metrics({
        "test_error_rate": len(err) / len(X_test_city),
        "test_errors_total": len(err)})

print("Анализ ошибок сохранён в MLflow (папка error_analysis) + метрика test_error_rate")

🏃 View run final_smart_negatives at: http://localhost:5000/#/experiments/2/runs/1826b14df2e94625aaf95f55c4260603
🧪 View experiment at: http://localhost:5000/#/experiments/2
Анализ ошибок сохранён в MLflow (папка error_analysis) + метрика test_error_rate


## Сравнение с baseline (пункт 7)
Сравнение финальной модели с простыми альтернативами (Dummy / эвристика по одной фиче / LogReg) на одном тестовом городе

In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 1) Dummy (мажоритарный)
dummy = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
p_dummy = dummy.predict_proba(X_test_city)[:, 1]

# 2) Чисто эвристика, ранжируем по total_poi_500m (без обучения)
p_heur = X_test_city["total_poi_500m"].values.astype(float)

# 3) Логрег
logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
).fit(X_tr, y_tr)
p_logreg = logreg.predict_proba(X_test_city)[:, 1]

# 4) CatBoost (наша модель)
p_cat = model_split.predict_proba(X_test_city)[:, 1]

rows = []
for name, p in [("Dummy (prior)", p_dummy),
                ("Эвристика: total_poi_500m", p_heur),
                ("LogReg", p_logreg),
                ("CatBoost (наша)", p_cat)]:
    roc = roc_auc_score(y_test_city, p)
    pr = average_precision_score(y_test_city, p)
    rows.append({"model": name, "roc_auc": round(roc, 4), "pr_auc": round(pr, 4)})

baseline_df = pd.DataFrame(rows)
baseline_df.to_csv("artifacts/baseline_comparison.csv", index=False)
print(baseline_df.to_string(index=False))
print(f"\nBaseline (доля позитивов на тесте): {y_test_city.mean():.4f}")

                    model  roc_auc  pr_auc
            Dummy (prior)   0.5000  0.5304
Эвристика: total_poi_500m   0.7454  0.7385
                   LogReg   0.7570  0.7777
          CatBoost (наша)   0.8564  0.8606

Baseline (доля позитивов на тесте): 0.5304


### График и логирование baseline (пункт 7)

In [10]:
fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(baseline_df))
w = 0.38
ax.bar([i - w/2 for i in x], baseline_df["roc_auc"], width=w, label="ROC-AUC")
ax.bar([i + w/2 for i in x], baseline_df["pr_auc"], width=w, label="PR-AUC")
ax.set_xticks(list(x)); ax.set_xticklabels(baseline_df["model"], rotation=20, ha="right")
ax.set_ylim(0, 1); ax.set_title(f"Baseline vs CatBoost (test = {TEST_CITY})"); ax.legend()
fig.tight_layout(); fig.savefig("artifacts/baseline_comparison.png", dpi=120); plt.close(fig)

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact("artifacts/baseline_comparison.csv", artifact_path="baseline")
    mlflow.log_artifact("artifacts/baseline_comparison.png", artifact_path="baseline")
    for _, r in baseline_df.iterrows():
        tag = r["model"].split(":")[0].split("(")[0].strip().replace(" ", "_").lower()
        mlflow.log_metric(f"baseline_{tag}_pr_auc", r["pr_auc"])

print("Baseline-сравнение сохранено в MLflow (папка baseline)")

🏃 View run final_smart_negatives at: http://localhost:5000/#/experiments/2/runs/1826b14df2e94625aaf95f55c4260603
🧪 View experiment at: http://localhost:5000/#/experiments/2
Baseline-сравнение сохранено в MLflow (папка baseline)


## Проверка устойчивости / robustness (пункт 8)
Вносим небольшой шум во входные данные (1/5/10%) и смотрим, как меняются предсказания

In [11]:
rng = np.random.default_rng(SEED)
base_proba = model_split.predict_proba(X_test_city)[:, 1]
base_pred = (base_proba >= 0.5).astype(int)

rob_rows = []
for noise in [0.01, 0.05, 0.10]:
    # мультипликативный шум: каждое значение * (1 + N(0, noise))
    factor = rng.normal(1.0, noise, size=X_test_city.shape)
    X_pert = X_test_city.values * factor
    X_pert = pd.DataFrame(X_pert, columns=X_test_city.columns, index=X_test_city.index)

    p = model_split.predict_proba(X_pert)[:, 1]
    pred = (p >= 0.5).astype(int)
    rob_rows.append({
        "noise_level": noise,
        "mean_abs_delta_proba": float(np.mean(np.abs(p - base_proba))),
        "flip_rate": float(np.mean(pred != base_pred)),
        "proba_corr": float(np.corrcoef(p, base_proba)[0, 1])})

rob_df = pd.DataFrame(rob_rows)
rob_df.to_csv("artifacts/robustness.csv", index=False)
print(rob_df.to_string(index=False))

 noise_level  mean_abs_delta_proba  flip_rate  proba_corr
        0.01              0.005343   0.007028    0.999339
        0.05              0.017449   0.022737    0.995594
        0.10              0.030223   0.034725    0.987307


### График и логирование robustness (пункт 8)

In [12]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(rob_df["noise_level"], rob_df["mean_abs_delta_proba"], "o-", label="mean |delta proba|")
ax1.plot(rob_df["noise_level"], rob_df["flip_rate"], "s-", label="flip rate")
ax1.set_xlabel("уровень шума"); ax1.set_ylabel("изменение"); ax1.legend(loc="upper left")
ax1.set_title("Robustness: реакция предсказаний на шум во входах")
fig.tight_layout(); fig.savefig("artifacts/robustness.png", dpi=120); plt.close(fig)

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact("artifacts/robustness.csv", artifact_path="robustness")
    mlflow.log_artifact("artifacts/robustness.png", artifact_path="robustness")
    for _, r in rob_df.iterrows():
        lvl = int(r["noise_level"] * 100)
        mlflow.log_metric(f"robustness_flip_rate_{lvl}pct", r["flip_rate"])
        mlflow.log_metric(f"robustness_mean_abs_delta_{lvl}pct", r["mean_abs_delta_proba"])

print("Robustness сохранён в MLflow (папка robustness)")

🏃 View run final_smart_negatives at: http://localhost:5000/#/experiments/2/runs/1826b14df2e94625aaf95f55c4260603
🧪 View experiment at: http://localhost:5000/#/experiments/2
Robustness сохранён в MLflow (папка robustness)


## Фиксация финальной модели с тегом/алиасом PRD (пункт 1)
Помечаем версию модели в Model Registry тегом `PRD=true` и алиасом `PRD`, это фиксирует, какая версия считается финальной (продовой). Загрузка по PRD и тестовый предикт выполняются в отдельном блокноте `02_load_prd_and_predict.ipynb` (пункт 9)

In [13]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# находим версию модели, обученную в нашем ране (на случай, если версий уже несколько)
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
ours = [v for v in versions if v.run_id == RUN_ID]
version = ours[0].version if ours else max(int(v.version) for v in versions)
print("Помечаем версию:", version)

# тег PRD на версии (требование задания)
client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "PRD", "true")
# поясняющие теги
client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "cv_roc_auc", f"{np.mean(cv_test_roc):.4f}")
client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "cv_pr_auc", f"{np.mean(cv_test_pr):.4f}")
client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "strategy", "smart_negatives")
# алиас PRD
client.set_registered_model_alias(REGISTERED_MODEL_NAME, "PRD", version)

# отметим и сам ран как PRD
with mlflow.start_run(run_id=RUN_ID):
    mlflow.set_tag("PRD", "true")

# проверка
mv = client.get_model_version(REGISTERED_MODEL_NAME, version)
print("Теги версии:", mv.tags)
print("Алиасы версии:", mv.aliases)
print(f"\nГотово, модель грузится как models:/{REGISTERED_MODEL_NAME}@PRD")

Помечаем версию: 1
🏃 View run final_smart_negatives at: http://localhost:5000/#/experiments/2/runs/1826b14df2e94625aaf95f55c4260603
🧪 View experiment at: http://localhost:5000/#/experiments/2
Теги версии: {'PRD': 'true', 'cv_roc_auc': '0.8733', 'cv_pr_auc': '0.8698', 'strategy': 'smart_negatives'}
Алиасы версии: ['PRD']

Готово, модель грузится как models:/atm_placement_catboost@PRD
